In [12]:
import numpy as np

import torch
import torch.optim as optim
from torch.nn import CTCLoss
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

from IPython.display import Audio, display
from tqdm import trange

In [13]:
def preprocess_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    # wav2vec2 expects 16kHz audio
    if sample_rate != required_sampling_rate:
        waveform = torchaudio.transforms.Resample(sample_rate, required_sampling_rate)(waveform)
    
    # mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    return waveform

In [14]:
def play_audio(waveform):
    display(Audio(waveform, rate=required_sampling_rate))

In [15]:
def transcribe(waveform):

    with torch.no_grad():
        logits = model(waveform).logits
    
    pred_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(pred_ids[0])
    
    return transcription

In [16]:
def reverse_audio(waveform):
    reverse = torch.flip(waveform, dims = [1])

    return reverse

In [17]:
def select_target():
    with open('inputs/targets.txt', mode='r', encoding='utf-8') as file:
        targets = file.read()
        
    targets = targets.split('\n')
    
    random_number_generator = np.random.default_rng()
    target = random_number_generator.choice(targets, size=1)[0].upper()
        
    return target

In [18]:
def CW_outter(waveform, target):
    
    yet_to_converge = True
    c = initial_constant
    lowerbound_c = 0
    upperbound_c = 1e9
    best_adverserial_audio = waveform
    for step in range(binary_search_steps):
        adverserial_audio, converged = CW_inner(waveform, target, c)
        if (converged == True):
            yet_to_converge = False
            upperbound_c = c 
            c = (lowerbound_c + upperbound_c)/2
            best_adverserial_audio = adverserial_audio
        elif (converged == False) and (yet_to_converge == False):
            lowerbound_c = c 
            c = (lowerbound_c + upperbound_c)/2
        elif (converged == False) and (yet_to_converge == True):
            c *= 10
            lowerbound_c = c 
    
    return best_adverserial_audio

In [19]:
def CW_inner(waveform, target, c):     
    # 1. Prepare target tokens
    target_tokens = processor(text=target, return_tensors="pt").input_ids[0].to(device)
    target_length = torch.tensor([len(target_tokens)], dtype=torch.long).to(device)
    
    # 2. Normalize the original audio (matching Wav2Vec2's typical input style)
    # Note: If your pipeline handles normalization inside the forward pass, adjust accordingly
    x = waveform.clone().detach().to(device)# Shape: [1, seq_len]
    w = torch.atanh((1 - 1e-6) * (waveform))

    w_perturbations = torch.zeros_like(w, requires_grad=True, device=device)
    optimizer = optim.Adam([w_perturbations], lr=lr)
    converged = False

    progress_bar = trange(max_iterations)
    for iteration in progress_bar:
        optimizer.zero_grad()
        
        # The adversarial audio sample
        adversarial_audio = torch.tanh(w + w_perturbations)
        
        # Forward pass through Wav2Vec2 model to get raw logits
        # Depending on your architecture, you may need to compute features first
        logits = model(adversarial_audio).logits  # Shape: [1, frame_steps, vocab_size]
        
        # Logits formatting for PyTorch CTCLoss: [frame_steps, batch_size, vocab_size]
        logits_log_probs = logits.log_softmax(2).transpose(0, 1)
        input_length = torch.tensor([logits_log_probs.size(0)], dtype=torch.long).to(device)
        
        # Calculate ASR Target Loss (CTC Loss)
        loss_ctc = ctc_loss_fn(logits_log_probs, target_tokens, input_length, target_length)
        
        # Calculate Distortion Loss (L2 Norm of the perturbation)
        loss_distortion = torch.mean(w_perturbations ** 2)
        
        # Total Loss formulation
        total_loss = loss_distortion + (c * loss_ctc)
        
        # Backward pass to find gradients w.r.t 'delta'
        total_loss.backward()
        optimizer.step()
        
        predicted_ids = torch.argmax(logits, dim=-1)
        current_transcription = processor.batch_decode(predicted_ids)[0]
        #status_bar.set_description_str(f"Transcript: {current_transcription}")
        
        progress_bar.set_postfix(
            c=f"{c}",
            Loss=f"{total_loss.item():.4f}",
            L2=f"{loss_distortion.item():.4f}"
        )
                   
        # Dynamic scaling (if target is reached, focus more on minimizing distortion)
        if current_transcription == target:
            converged = True
            best_adverserial_example = torch.tanh(w + w_perturbations).detach().cpu().squeeze(0)
            print("--> Success! Target achieved. Refining imperceptibility...")
            break
        
        best_adverserial_example = torch.tanh(w + w_perturbations).detach().cpu().squeeze(0)
        
    return best_adverserial_example, converged

In [20]:
# load model and its utilities
device = 'cuda' if torch.cuda.is_available() else 'cpu' 

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(device)

required_sampling_rate = processor.feature_extractor.sampling_rate
ctc_loss_fn = CTCLoss(blank=processor.tokenizer.pad_token_id, zero_infinity=True)

/opt/anaconda3/envs/py311_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/wav2vec2-base-960h were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2

In [21]:
# process audio
audio_path = 'inputs/sample.ogg'

forward_waveform = preprocess_audio(audio_path)
forward_transcription = transcribe(forward_waveform)
play_audio(forward_waveform)
print(forward_transcription)

reverse_waveform = reverse_audio(forward_waveform)
reverse_transcription = transcribe(reverse_waveform)
play_audio(reverse_waveform)
print(reverse_transcription)

THE QUICK BROWN FOX JUMPS OVER THE LAZY DOG


I NESY A LETTER OF OICE MY SCOFF MYR BELEN


In [ ]:
# perform Carlini-Wagner attack
binary_search_steps = 5
initial_constant = 0.0001
lr = 0.01
max_iterations = 500

target = select_target()
print(target)
adversarial_waveform = CW_outter(reverse_waveform, target)
converged = (adversarial_waveform != reverse_waveform).any().item()
play_audio(adversarial_waveform)

SATAN REPRESENTS VENGEANCE INSTEAD OF TURNING THE OTHER CHEEK


 69%|████▊  | 344/500 [10:28<04:52,  1.87s/it, L2=0.0014, Loss=0.0015, c=0.0001]